In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


def deduplicate_latest(
    df,
    key_columns,
    order_column,
    descending=True
):
    """
    Keep the latest record for each business key.
    """

    order_expression = (
        F.col(order_column).desc()
        if descending
        else F.col(order_column).asc()
    )

    window_spec = (
        Window
        .partitionBy(*key_columns)
        .orderBy(order_expression)
    )

    return (
        df
        .withColumn("_dedup_rank", F.row_number().over(window_spec))
        .filter(F.col("_dedup_rank") == 1)
        .drop("_dedup_rank")
    )